# 15. SCENIC+ Enhancer → Consensus Peak ID Mapping

**Goal:** Map enhancer regions identified by SCENIC+ eRegulons back to our cross-species consensus peak IDs (v3). This enables:
- Linking eRegulon enhancers to the unified peak nomenclature
- Classifying enhancers as unified (conserved) vs species-specific
- Cross-run robustness analysis (comparing multiple seeds)

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import pyranges as pr
import mudata as mu
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## Configuration

In [ ]:
BASE_DIR = "/home/jjanssens/jjans/analysis/adult_intestine"
SCENICPLUS_DIR = os.path.join(BASE_DIR, "scenicplus")
CONSENSUS_DIR = os.path.join(BASE_DIR, "peaks/cross_species_consensus_v3/10_final")

SPECIES_LIST = ["Human", "Gorilla", "Chimpanzee", "Bonobo", "Macaque", "Marmoset"]

# Define SCENIC+ run directories to analyze.
# Each entry: (species, run_label, path_to_scplusmdata.h5mu)
# Edit this to match your completed runs.
RUNS = []
for species in SPECIES_LIST:
    # Check for existing completed runs (original + seed variants)
    # Original runs
    orig_mdata = os.path.join(SCENICPLUS_DIR, f"scplus_pipeline_{species}", "Snakemake", "scplusmdata.h5mu")
    if os.path.exists(orig_mdata):
        RUNS.append((species, "original", orig_mdata))
    
    # Seed/downsample variants
    pattern = os.path.join(SCENICPLUS_DIR, f"scplus_pipeline_{species}_seed*", "Snakemake", "scplusmdata.h5mu")
    for p in sorted(glob.glob(pattern)):
        run_label = Path(p).parent.parent.name.replace(f"scplus_pipeline_{species}_", "")
        RUNS.append((species, run_label, p))

# Output directory
OUTPUT_DIR = os.path.join(BASE_DIR, "peaks/cross_species_consensus_v3/scenic_enhancer_mapping")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Found {len(RUNS)} completed SCENIC+ runs:")
for species, label, path in RUNS:
    print(f"  {species} / {label}")

## Helper functions

In [ ]:
def load_consensus_peaks(species):
    """Load consensus peaks as PyRanges with peak IDs."""
    path = os.path.join(CONSENSUS_DIR, f"all_peaks_{species}.bed")
    df = pd.read_csv(path, sep="\t", header=None,
                     names=["Chromosome", "Start", "End", "peak_id"])
    return pr.PyRanges(df)


def extract_eregulon_regions(mdata_path):
    """
    Extract enhancer regions from SCENIC+ MuData eRegulon metadata.
    Returns DataFrame with columns: TF, region, target_gene, annotation_type.
    """
    mdata = mu.read(mdata_path)
    
    rows = []
    for annot_type in ["direct", "extended"]:
        key = f"{annot_type}_e_regulon_metadata"
        if key not in mdata.uns:
            print(f"  WARNING: {key} not found in MuData.")
            continue
        
        ereg_df = mdata.uns[key]
        if isinstance(ereg_df, pd.DataFrame):
            # Expected columns: TF, Region, Gene (may vary)
            for col_region in ["Region", "region", "regions"]:
                if col_region in ereg_df.columns:
                    break
            else:
                print(f"  WARNING: No region column found in {key}. Columns: {ereg_df.columns.tolist()}")
                continue
            
            for col_tf in ["TF", "tf", "Transcription_factor"]:
                if col_tf in ereg_df.columns:
                    break
            else:
                col_tf = ereg_df.columns[0]
            
            for col_gene in ["Gene", "gene", "target_gene"]:
                if col_gene in ereg_df.columns:
                    break
            else:
                col_gene = None
            
            for _, row in ereg_df.iterrows():
                rows.append({
                    "TF": row[col_tf],
                    "region": row[col_region],
                    "target_gene": row[col_gene] if col_gene else None,
                    "annotation_type": annot_type,
                })
    
    del mdata
    return pd.DataFrame(rows)


def regions_to_pyranges(region_series):
    """Convert 'chr1:12345-67890' style regions to PyRanges."""
    parsed = region_series.str.extract(r"^(.+?):(\d+)-(\d+)$")
    parsed.columns = ["Chromosome", "Start", "End"]
    parsed["Start"] = parsed["Start"].astype(int)
    parsed["End"] = parsed["End"].astype(int)
    parsed["region"] = region_series.values
    return pr.PyRanges(parsed.dropna(subset=["Chromosome"]))


def map_regions_to_consensus(region_pr, consensus_pr):
    """Map regions to overlapping consensus peak IDs."""
    joined = region_pr.join(consensus_pr, how="left")
    jdf = joined.df
    if len(jdf) == 0:
        return pd.DataFrame(columns=["region", "peak_id"])
    result = jdf[["region", "peak_id"]].drop_duplicates()
    # pyranges 0.0.x uses -1 for unmatched left joins
    result = result[~result["peak_id"].isin([-1, "-1"])]
    return result

## Load consensus peaks

In [ ]:
consensus_peaks = {}
for species in SPECIES_LIST:
    consensus_peaks[species] = load_consensus_peaks(species)
    print(f"{species}: {len(consensus_peaks[species]):,} consensus peaks")

## Process each SCENIC+ run

In [ ]:
all_run_results = {}
run_stats = []

for species, run_label, mdata_path in RUNS:
    print(f"\n{'='*60}")
    print(f"  {species} / {run_label}")
    print(f"{'='*60}")
    
    # Extract eRegulon regions
    ereg_df = extract_eregulon_regions(mdata_path)
    if len(ereg_df) == 0:
        print("  No eRegulon regions found, skipping.")
        continue
    
    print(f"  eRegulon entries: {len(ereg_df):,}")
    print(f"  Unique TFs: {ereg_df['TF'].nunique()}")
    print(f"  Unique regions: {ereg_df['region'].nunique():,}")
    
    # Get unique regions and map to consensus
    unique_regions = ereg_df["region"].unique()
    region_pr = regions_to_pyranges(pd.Series(unique_regions))
    cons = consensus_peaks[species]
    
    mapping = map_regions_to_consensus(region_pr, cons)
    
    # Classify peak types
    if len(mapping) > 0:
        mapping["peak_type"] = mapping["peak_id"].apply(
            lambda x: "unified" if str(x).startswith("unified_") else
                      "human_specific" if str(x).startswith("human_peak_") else
                      "species_specific"
        )
    
    # Merge mapping back into eRegulon table
    ereg_annotated = ereg_df.merge(mapping, on="region", how="left")
    
    # Stats
    n_unique = len(unique_regions)
    n_mapped = mapping["region"].nunique() if len(mapping) > 0 else 0
    pct_mapped = n_mapped / n_unique * 100 if n_unique > 0 else 0
    
    print(f"\n  Mapping results:")
    print(f"    Unique enhancer regions: {n_unique:,}")
    print(f"    Mapped to consensus:     {n_mapped:,} ({pct_mapped:.1f}%)")
    print(f"    Unmapped:                {n_unique - n_mapped:,}")
    
    if len(mapping) > 0:
        pt_counts = mapping["peak_type"].value_counts()
        print(f"    Peak type breakdown (unique regions):")
        for pt, cnt in pt_counts.items():
            print(f"      {pt}: {cnt:,}")
    
    # Save annotated eRegulon table
    out_path = os.path.join(OUTPUT_DIR, f"eRegulon_peaks_annotated_{species}_{run_label}.tsv")
    ereg_annotated.to_csv(out_path, sep="\t", index=False)
    print(f"  Saved: {out_path}")
    
    run_key = f"{species}_{run_label}"
    all_run_results[run_key] = {
        "ereg_annotated": ereg_annotated,
        "mapping": mapping,
        "unique_regions": set(unique_regions),
    }
    
    run_stats.append({
        "species": species,
        "run_label": run_label,
        "n_eregulon_entries": len(ereg_df),
        "n_unique_regions": n_unique,
        "n_unique_TFs": ereg_df["TF"].nunique(),
        "n_mapped": n_mapped,
        "pct_mapped": pct_mapped,
    })

stats_df = pd.DataFrame(run_stats)
print("\nProcessing complete.")

## Summary across runs

In [ ]:
if len(stats_df) > 0:
    display(stats_df.style.format({"pct_mapped": "{:.1f}%"}))
else:
    print("No completed runs found.")

## Per-TF conservation analysis

For each TF's enhancer regions, what fraction are unified (cross-species conserved) vs species-specific?

In [ ]:
for run_key, result in all_run_results.items():
    ereg = result["ereg_annotated"]
    if "peak_type" not in ereg.columns or ereg["peak_type"].isna().all():
        continue
    
    print(f"\n--- {run_key} ---")
    
    # Per-TF: count unique regions by peak type
    tf_stats = (
        ereg.dropna(subset=["peak_id"])
        .drop_duplicates(subset=["TF", "region", "peak_type"])
        .groupby(["TF", "peak_type"])
        .size()
        .unstack(fill_value=0)
    )
    
    if "unified" in tf_stats.columns:
        tf_stats["total"] = tf_stats.sum(axis=1)
        tf_stats["pct_unified"] = (tf_stats["unified"] / tf_stats["total"] * 100).round(1)
        tf_stats = tf_stats.sort_values("pct_unified", ascending=False)
        
        print(f"  Top 15 TFs by % unified enhancers:")
        display(tf_stats.head(15))
        
        # Plot distribution
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(tf_stats["pct_unified"], bins=20, edgecolor="white")
        ax.set_xlabel("% unified (conserved) enhancers per TF")
        ax.set_ylabel("Number of TFs")
        ax.set_title(f"{run_key}: TF enhancer conservation")
        ax.axvline(50, ls="--", color="red", alpha=0.7)
        plt.tight_layout()
        plt.show()

## Cross-run robustness analysis (multiple seeds)

Compare eRegulon region sets across different seeds for the same species — how reproducible are the identified enhancers?

In [ ]:
# Group runs by species
from collections import defaultdict
species_runs = defaultdict(list)
for run_key, result in all_run_results.items():
    species = run_key.split("_")[0]
    species_runs[species].append((run_key, result))

for species, runs in species_runs.items():
    if len(runs) < 2:
        continue
    
    print(f"\n{'='*60}")
    print(f"  {species}: comparing {len(runs)} runs")
    print(f"{'='*60}")
    
    # Jaccard similarity of region sets
    run_keys = [rk for rk, _ in runs]
    region_sets = {rk: result["unique_regions"] for rk, result in runs}
    
    jaccard_matrix = pd.DataFrame(index=run_keys, columns=run_keys, dtype=float)
    for rk1, rk2 in combinations(run_keys, 2):
        s1, s2 = region_sets[rk1], region_sets[rk2]
        intersection = len(s1 & s2)
        union = len(s1 | s2)
        j = intersection / union if union > 0 else 0.0
        jaccard_matrix.loc[rk1, rk2] = j
        jaccard_matrix.loc[rk2, rk1] = j
    for rk in run_keys:
        jaccard_matrix.loc[rk, rk] = 1.0
    
    print("\nJaccard similarity of enhancer region sets:")
    display(jaccard_matrix.style.format("{:.3f}").background_gradient(cmap="YlGn", vmin=0, vmax=1))
    
    # Regions found in ALL runs
    all_region_sets = [region_sets[rk] for rk in run_keys]
    consensus_regions = set.intersection(*all_region_sets)
    any_regions = set.union(*all_region_sets)
    print(f"\n  Regions in ALL runs: {len(consensus_regions):,}")
    print(f"  Regions in ANY run:  {len(any_regions):,}")
    print(f"  Reproducibility:     {len(consensus_regions)/len(any_regions):.1%}" if any_regions else "")

## Overall summary

In [ ]:
if len(stats_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Mapping rate per run
    ax = axes[0]
    sns.barplot(data=stats_df, x="species", y="pct_mapped", hue="run_label",
                ax=ax, palette="Set2")
    ax.axhline(90, ls="--", color="red", alpha=0.7)
    ax.set_ylabel("% enhancers mapped to consensus")
    ax.set_title("eRegulon enhancer → consensus peak mapping")
    ax.set_ylim(0, 105)
    
    # Plot 2: Number of enhancers per run
    ax = axes[1]
    sns.barplot(data=stats_df, x="species", y="n_unique_regions", hue="run_label",
                ax=ax, palette="Set2")
    ax.set_ylabel("Unique enhancer regions")
    ax.set_title("Number of eRegulon enhancers per run")
    
    plt.tight_layout()
    plt.show()

print("\nAnnotated eRegulon files saved to:", OUTPUT_DIR)
print("Done.")